In [9]:
import os
import pandas as pd
import random
import matplotlib.pyplot as plt
from ipywidgets import widgets, VBox, HBox, Layout, Output, HTML
from IPython.display import display, clear_output
import json
import math

# Path to the data folder
folder_path = 'D:\\code\\uom_explore\\raw_data\\2024_09_03\\de_brujin_large_even_1'

# List all CSV files in the folder, excluding those with '_BME680' in their names
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv') and '_BME680' not in f]

# Randomly select files (up to 20 for this example)
selected_files = random.sample(csv_files, min(20, len(csv_files)))

# Default column names
default_columns = ['Setting', 'Timestamp', 'sensor_value']

# Dictionary to store the responses, initialized with 'Uncertain'
responses = {file: 'Uncertain' for file in selected_files}

# Function to process and plot a single file
def process_and_plot_file(file, ax):
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path, sep=',', names=default_columns, header=0)
    df['Normalized_Timestamp'] = df['Timestamp'] - df['Timestamp'].iloc[0]
    
    for setting, group in df.groupby('Setting'):
        ax.plot(group['Normalized_Timestamp'], group['sensor_value'], marker='o', linestyle='-', markersize=2)
    
    ax.set_title(f'{file}', fontsize=10)
    ax.set_xlabel('Normalized Timestamp', fontsize=8)
    ax.set_ylabel('Sensor Value', fontsize=8)
    ax.tick_params(axis='both', which='major', labelsize=6)
    ax.grid(True)

# Function to create a page of plots and responses
def create_page(files):
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    fig.tight_layout(pad=4.0)
    
    radio_buttons = []
    for i, file in enumerate(files):
        process_and_plot_file(file, axs[i])
        radio = widgets.RadioButtons(
            options=['OK', 'NG', 'Uncertain'],
            value=responses[file],  # Set the initial value to the current response
            description=f'Quality:',
            disabled=False,
            layout=Layout(width='auto')
        )
        radio.observe(lambda change, file=file: update_response(change, file), names='value')
        radio_buttons.append(VBox([HTML(f'<div style="text-align: center;">{file}</div>'), radio], layout=Layout(align_items='center')))
    
    # Remove any unused subplots
    for j in range(len(files), 4):
        fig.delaxes(axs[j])
    
    plt.show()
    display(HBox(radio_buttons, layout=Layout(justify_content='center')))

# Function to update response when radio button is changed
def update_response(change, file):
    responses[file] = change['new']
    save_responses()

# Function to save responses
def save_responses():
    with open('quality_check_report.json', 'w') as f:
        json.dump(responses, f, indent=4)
    print("Responses saved to quality_check_report.json")

# Create navigation buttons
prev_button = widgets.Button(description="< Previous")
next_button = widgets.Button(description="Next >")
submit_button = widgets.Button(description="Submit All Responses")
page_indicator = widgets.HTML()
nav_buttons = HBox([prev_button, page_indicator, next_button])

# Create an output widget to hold the current page
out = Output()

# Calculate the number of pages
files_per_page = 4
n_pages = math.ceil(len(selected_files) / files_per_page)
current_page = [0]  # Using a list to make it mutable in nested functions

# Function to update the page
def update_page(page):
    with out:
        clear_output(wait=True)
        start_idx = page * files_per_page
        end_idx = min(start_idx + files_per_page, len(selected_files))
        create_page(selected_files[start_idx:end_idx])
        page_indicator.value = f'<div style="text-align: center;">Page {page + 1} of {n_pages}</div>'
        if page == n_pages - 1:  # If it's the last page
            display(HBox([prev_button, page_indicator, submit_button]))
        else:
            display(nav_buttons)

# Navigation button callbacks
def on_prev_click(b):
    current_page[0] = max(0, current_page[0] - 1)
    update_page(current_page[0])

def on_next_click(b):
    current_page[0] = min(n_pages - 1, current_page[0] + 1)
    update_page(current_page[0])

prev_button.on_click(on_prev_click)
next_button.on_click(on_next_click)

# Submit button callback
def on_submit_click(b):
    save_responses()
    print("All responses submitted and saved to quality_check_report.json")

submit_button.on_click(on_submit_click)

# Initial page display
display(out)
update_page(0)

Output()

Responses saved to quality_check_report.json
Responses saved to quality_check_report.json
Responses saved to quality_check_report.json
Responses saved to quality_check_report.json


Responses saved to quality_check_report.json
All responses submitted and saved to quality_check_report.json
